# Ramsey R(3,10) V8 — Google Colab Multi-Core Proof-First Solver
Solves all 12 surviving canonical `D9-C6` $3\times 3$ skeleton orbits (and `D9-C5` / `NA-T1` / `NA-T2` slices) using V8 Proof-First Kissat + `drat-trim` + `lrat-check`. Pinned to `f4b97c55abbdeb835fc013771e9643b703d0526c`.

In [ ]:
import os, sys, subprocess, shutil, json
from pathlib import Path

CASE = os.environ.get('RAMSEY_D9_CASE', 'C6')
COMMIT = 'f4b97c55abbdeb835fc013771e9643b703d0526c'
WORK = Path('/content/ramsey_v8_out') if Path('/content').exists() else Path('/tmp/ramsey_v8_out')
TMP = Path('/tmp/ramsey-d9-v8-colab')
WORK.mkdir(parents=True, exist_ok=True)
TMP.mkdir(parents=True, exist_ok=True)

def run(args, cwd=None):
    print('+', ' '.join(map(str, args)), flush=True)
    subprocess.run(list(map(str, args)), cwd=cwd, check=True)

def fetch_repo(name, url, rev):
    d = TMP / name
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True)
    run(['git', 'init', '-q'], d)
    run(['git', 'remote', 'add', 'origin', url], d)
    run(['git', 'fetch', '-q', '--depth', '1', 'origin', rev], d)
    run(['git', 'checkout', '-q', '--detach', 'FETCH_HEAD'], d)
    assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=d, text=True).strip() == rev
    return d

print('CPU count:', os.cpu_count(), 'Work dir:', WORK, flush=True)
repo = fetch_repo('repo', 'https://github.com/davidlueng986-alt/math-open-to-closed-compute.git', COMMIT)
run([
    sys.executable, '-u',
    str(repo / 'jobs/ramsey-r310-v8-w2-d9c345/run_cloud_v8.py'),
    '--case', CASE,
    '--workers', str(max(1, min(4, os.cpu_count() or 2))),
    '--wall-budget-seconds', '18000',
    '--per-orbit-seconds', '900',
    '--auto-build-tools',
    '--tools-dir', str(TMP / 'tools'),
    '--out', str(WORK / f'd9-{CASE.lower()}-v8'),
])
